# Bocetos gratis de personajes con FLUX.1 schnell

Genera imágenes de prueba de un personaje **sin gastar créditos de Higgsfield**, en la GPU gratuita de Google Colab.
Sirve para **definir** el personaje (silueta, colores, ropa, actitud). La versión final se produce después en Higgsfield.

- Modelo: **FLUX.1 [schnell]** de Black Forest Labs, licencia **Apache-2.0** (permite uso comercial).
- Va comprimido a 4 bits (NF4) para que entre en la GPU T4 gratuita.
- Solo personajes **originales**: nada de personajes existentes ni personas reales que no hayan dado permiso.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

> No probado todavía en una sesión real de Colab (2026-11-07). Si algo falla, anota el error y pásaselo a Claude.


In [ ]:
# 1. Instalar (tarda 1-2 minutos)
!pip -q install -U diffusers transformers accelerate bitsandbytes sentencepiece protobuf
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cargar el modelo
La primera vez descarga unos 30 GB y tarda varios minutos. Si Colab se reinicia por falta de memoria RAM, usa la celda
**2b** en su lugar.

In [ ]:
import torch, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TBnb

REPO = "black-forest-labs/FLUX.1-schnell"
dtype = torch.float16  # la T4 no tiene bfloat16 rápido

transformer = FluxTransformer2DModel.from_pretrained(
    REPO, subfolder="transformer", torch_dtype=dtype,
    quantization_config=DBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype),
)
text_encoder_2 = T5EncoderModel.from_pretrained(
    REPO, subfolder="text_encoder_2", torch_dtype=dtype,
    quantization_config=TBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype),
)
pipe = FluxPipeline.from_pretrained(REPO, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
gc.collect(); torch.cuda.empty_cache()
print("Modelo listo")

### 2b. Solo si la celda 2 se quedó sin memoria
Usa un paquete ya comprimido por la comunidad (menos RAM al cargar). Reinicia el entorno, corre la celda 1 y luego esta.

In [ ]:
# import torch
# from diffusers import FluxPipeline
# pipe = FluxPipeline.from_pretrained("vftfnn/flux1-schnell-nf4", torch_dtype=torch.float16)
# pipe.enable_model_cpu_offload()
# print("Modelo listo (versión comprimida de la comunidad)")

## 3. Pegar el prompt y generar
Pega el **prompt** que entrega el afinador de personajes (`creador-personajes/afinado/<personaje>.md`).

- `SEMILLAS`: cada número es una variante. **Repetir la misma semilla** con un prompt cambiado muestra solo el efecto
  del cambio (así se compara A contra B).
- Formato vertical 9:16 (576 × 1024). Cada imagen tarda unos segundos a un minuto en la T4.

In [ ]:
PERSONAJE = "spike"
PROMPT = """Realistic photo of an original, wildly exaggerated eccentric character named Spike, larger-than-life comedic caricature proportions: absurdly tall and stick-thin with extra-long arms, his signature hair is a giant crown of rigid punk liberty spikes, about a dozen spikes each as long as his forearm, entirely glowing NEON ORANGE, fluorescent safety-orange, ultra saturated, perfectly stiff and glossy. Bright green expressive eyes, thick bleached eyebrows, a wide cheeky full-toothed grin, chewing bubblegum. Oversized shiny purple velvet tracksuit with enormous padded shoulders, a fluffy neon-yellow feather boa, chunky rings on every finger, huge white platform sneakers, dramatic rock-star pose. Seamless white infinity studio, soft even light, full body, vertical 9:16."""
SEMILLAS = [1, 2, 3, 4]

import os, torch
from IPython.display import display
os.makedirs(f"/content/bocetos/{PERSONAJE}", exist_ok=True)
imagenes = []
for s in SEMILLAS:
    img = pipe(PROMPT, width=576, height=1024, num_inference_steps=4, guidance_scale=0.0,
               max_sequence_length=256, generator=torch.Generator("cpu").manual_seed(s)).images[0]
    ruta = f"/content/bocetos/{PERSONAJE}/{PERSONAJE}_semilla{s}.png"
    img.save(ruta); imagenes.append(img); print("Guardada:", ruta)

# Mostrar en fila
from PIL import Image
fila = Image.new("RGB", (288 * len(imagenes), 512), "white")
for i, im in enumerate(imagenes): fila.paste(im.resize((288, 512)), (288 * i, 0))
display(fila)

## 4. Descargar
Baja las imágenes y muéstraselas a Claude con tu veredicto («me gusta la semilla 3, pero…»). El afinador anota qué
semilla y qué prompt funcionaron.

In [ ]:
!cd /content && zip -qr bocetos.zip bocetos
from google.colab import files
files.download("/content/bocetos.zip")

## 5. (Experimental) Probar movimiento con Wan 2.2
Wan 2.2 TI2V-5B (Apache-2.0) puede animar una imagen, pero en la T4 gratuita queda **muy justo de memoria** (las
fuentes no coinciden). Si quieres intentarlo, hay un cuaderno de la comunidad: https://github.com/theelderemo/wan2.2-google-colab
(elige el modo TI2V-5B, baja la resolución y la cantidad de cuadros si se queda sin memoria). Si no funciona, las
pruebas de movimiento se hacen en Higgsfield con Seedance 2.0 Mini en 480p (6 créditos por 12 s).